In [ ]:
# =============================================================================
# 步驟1: 套件安裝與匯入
# =============================================================================

print("\n📦 步驟1: 安裝必要套件...")

# 安裝因子分析和視覺化套件
!pip install factor_analyzer -q
!pip install plotly -q

In [ ]:
# =============================================================================
# Garmin運動手環睡眠因子分析 - 簡潔版本
# 學術用途 - 多變量分析期末報告
# =============================================================================

print("🧠 Garmin睡眠因子分析開始執行...")
print("="*70)

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
from factor_analyzer.factor_analyzer import calculate_kmo
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("✅ 套件安裝完成!")

# 讀取資料
print("\n📊 讀取資料...")
df = pd.read_csv('/content/drive/MyDrive/多變量專案/sleep_activity_merged_final.csv')
print(f"✅ 資料讀取成功! 資料形狀: {df.shape}")

# 選擇睡眠變數
sleep_variables = [
    'sleep_hours',
    'deep_sleep_ratio',
    'deepSleepDurationInSeconds',
    'lightSleepDurationInSeconds',
    'remSleepInSeconds',
    'durationInSeconds'
]

available_vars = [var for var in sleep_variables if var in df.columns]
print(f"可用睡眠變數: {available_vars}")

# 準備分析資料
sleep_data = df[available_vars].dropna()
print(f"有效記錄: {len(sleep_data)} 筆")

# 標準化
scaler = StandardScaler()
sleep_data_scaled = pd.DataFrame(
    scaler.fit_transform(sleep_data),
    columns=available_vars,
    index=sleep_data.index
)

print("\n🔬 因子分析適合性檢驗...")

# Bartlett球面檢驗
chi_square_value, p_value = calculate_bartlett_sphericity(sleep_data_scaled)
print(f"Bartlett檢驗: 卡方={chi_square_value:.3f}, p={p_value:.2e}")

# KMO檢驗
kmo_all, kmo_model = calculate_kmo(sleep_data_scaled)
print(f"KMO值: {kmo_model:.3f}")

# 顯示相關係數矩陣熱圖
correlation_matrix = sleep_data_scaled.corr()
fig_corr = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.index,
    colorscale='RdBu',
    zmid=0,
    text=correlation_matrix.round(3).values,
    texttemplate="%{text}",
    textfont={"size": 10}
))
fig_corr.update_layout(title="睡眠變數相關係數矩陣", width=700, height=600)
fig_corr.show()

print("\n📈 確定因子數量...")

# 初步因子分析獲取特徵值
fa_initial = FactorAnalyzer(rotation=None)
fa_initial.fit(sleep_data_scaled)
eigenvalues, _ = fa_initial.get_eigenvalues()

# 碎石圖
fig_scree = make_subplots(rows=1, cols=2, subplot_titles=["碎石圖", "累積解釋變異"])

# 碎石圖
fig_scree.add_trace(
    go.Scatter(x=list(range(1, len(eigenvalues)+1)), y=eigenvalues,
               mode='lines+markers', name='特徵值'),
    row=1, col=1
)
fig_scree.add_hline(y=1, line_dash="dash", line_color="red", row=1, col=1)

# 累積變異
cumulative_variance = np.cumsum(eigenvalues) / len(available_vars)
fig_scree.add_trace(
    go.Scatter(x=list(range(1, len(cumulative_variance)+1)), y=cumulative_variance,
               mode='lines+markers', name='累積變異'),
    row=1, col=2
)
fig_scree.add_hline(y=0.6, line_dash="dash", line_color="orange", row=1, col=2)

fig_scree.update_layout(title="因子數量確定分析", height=400)
fig_scree.show()

# 決定因子數量
recommended_factors = 2
kaiser_factors = sum(eigenvalues > 1)
print(f"Kaiser準則建議: {kaiser_factors} 個因子")
print(f"使用因子數: {recommended_factors} 個")

print("\n🔬 執行因子分析...")

# 執行因子分析 (使用兩種轉軸方法)
rotation_methods = ['varimax', 'oblimin']
fa_results = {}

for rotation in rotation_methods:
    try:
        fa = FactorAnalyzer(n_factors=recommended_factors, rotation=rotation, method='ml')
        fa.fit(sleep_data_scaled)
        fa_results[rotation] = fa
        print(f"✅ {rotation.upper()} 轉軸成功")
    except:
        try:
            fa = FactorAnalyzer(n_factors=recommended_factors, rotation=rotation, method='principal')
            fa.fit(sleep_data_scaled)
            fa_results[rotation] = fa
            print(f"✅ {rotation.upper()} 轉軸成功 (主成分法)")
        except Exception as e:
            print(f"❌ {rotation.upper()} 轉軸失敗: {e}")

# 選擇最佳結果
best_rotation = 'oblimin' if 'oblimin' in fa_results else list(fa_results.keys())[0]
fa_best = fa_results[best_rotation]

print(f"\n📊 {best_rotation.upper()} 轉軸結果:")

# 因子負載矩陣
loadings = fa_best.loadings_
loadings_df = pd.DataFrame(
    loadings,
    index=available_vars,
    columns=[f'因子{i+1}' for i in range(recommended_factors)]
)
print("因子負載矩陣:")
print(loadings_df.round(3))

# 因子負載熱圖
fig_loadings = go.Figure(data=go.Heatmap(
    z=loadings,
    x=[f'因子{i+1}' for i in range(recommended_factors)],
    y=available_vars,
    colorscale='RdBu',
    zmid=0,
    text=np.round(loadings, 3),
    texttemplate="%{text}",
    textfont={"size": 12}
))
fig_loadings.update_layout(title="因子負載矩陣熱圖", width=600, height=500)
fig_loadings.show()

# 共同性分析
communalities = fa_best.get_communalities()
communalities_df = pd.DataFrame({
    '變數': available_vars,
    '共同性': communalities
})

fig_comm = go.Figure()
fig_comm.add_trace(go.Bar(
    x=available_vars,
    y=communalities,
    text=np.round(communalities, 3),
    textposition='auto'
))
fig_comm.add_hline(y=0.5, line_dash="dash", line_color="green")
fig_comm.update_layout(title="變數共同性分析", xaxis_tickangle=45)
fig_comm.show()

print(f"\n共同性分析:")
for _, row in communalities_df.iterrows():
    print(f"  {row['變數']}: {row['共同性']:.3f}")

# 計算解釋變異
eigenvalues_rotated = np.sum(loadings**2, axis=0)
explained_variance = eigenvalues_rotated / len(available_vars)
total_explained = sum(explained_variance)

print(f"\n📈 解釋變異:")
for i, variance in enumerate(explained_variance):
    print(f"  因子{i+1}: {variance:.1%}")
print(f"  總計: {total_explained:.1%}")

# 因子間相關性 (斜交轉軸)
if hasattr(fa_best, 'phi_') and recommended_factors == 2:
    factor_correlation = fa_best.phi_[0, 1]
    print(f"\n🔗 因子間相關性: {factor_correlation:.3f}")

print("\n🔍 與PCA結果比較...")
if 'deep_sleep_ratio' in available_vars and 'sleep_hours' in available_vars:
    deep_idx = available_vars.index('deep_sleep_ratio')
    sleep_idx = available_vars.index('sleep_hours')

    print("PCA發現: deep_sleep_ratio(+0.720) vs sleep_hours(-0.667)")
    print("因子分析結果:")
    for i in range(recommended_factors):
        deep_loading = loadings[deep_idx, i]
        sleep_loading = loadings[sleep_idx, i]
        print(f"  因子{i+1}: deep_sleep_ratio({deep_loading:.3f}) vs sleep_hours({sleep_loading:.3f})")

        if (deep_loading * sleep_loading) < 0 and (abs(deep_loading) > 0.4 or abs(sleep_loading) > 0.4):
            print(f"    ✅ 驗證PCA發現的權衡關係!")

# 生成因子分數
factor_scores = fa_best.transform(sleep_data_scaled)
factor_scores_df = pd.DataFrame(
    factor_scores,
    columns=[f'睡眠因子{i+1}_分數' for i in range(recommended_factors)],
    index=sleep_data.index
)

print(f"\n📊 因子分數統計:")
print(factor_scores_df.describe().round(3))

# 因子分數分布圖
if recommended_factors == 2:
    fig_scores = go.Figure()
    fig_scores.add_trace(go.Scatter(
        x=factor_scores[:, 0],
        y=factor_scores[:, 1],
        mode='markers',
        marker=dict(size=6, opacity=0.6)
    ))
    fig_scores.update_layout(
        title="因子分數分布",
        xaxis_title="因子1分數",
        yaxis_title="因子2分數"
    )
    fig_scores.show()

print("\n💾 儲存結果到Google Drive...")

# 儲存分析結果
loadings_df.to_csv('/content/drive/MyDrive/多變量專案/sleep_factor_loadings.csv')
factor_scores_df.to_csv('/content/drive/MyDrive/多變量專案/sleep_factor_scores.csv')
communalities_df.to_csv('/content/drive/MyDrive/多變量專案/sleep_communalities.csv', index=False)

# 整合資料
sleep_data_with_factors = pd.concat([sleep_data, factor_scores_df], axis=1)
sleep_data_with_factors.to_csv('/content/drive/MyDrive/多變量專案/sleep_data_with_factor_scores.csv')

# 分析總結
summary_df = pd.DataFrame({
    '分析項目': ['資料筆數', '變數數量', '因子數量', 'Bartlett卡方值', 'KMO值', '總解釋變異'],
    '結果': [len(sleep_data), len(available_vars), recommended_factors,
            f"{chi_square_value:.3f}", f"{kmo_model:.3f}", f"{total_explained:.1%}"]
})
summary_df.to_csv('/content/drive/MyDrive/多變量專案/sleep_factor_analysis_summary.csv', index=False)

print("✅ 結果已儲存到Google Drive")
print("📁 生成檔案:")
print("  - sleep_factor_loadings.csv (因子負載矩陣)")
print("  - sleep_factor_scores.csv (因子分數)")
print("  - sleep_communalities.csv (共同性分析)")
print("  - sleep_data_with_factor_scores.csv (完整資料)")
print("  - sleep_factor_analysis_summary.csv (分析總結)")

print(f"\n🎉 睡眠因子分析完成!")
print(f"🎯 主要發現:")
print(f"  ✅ 成功萃取 {recommended_factors} 個睡眠因子")
print(f"  ✅ 總解釋變異: {total_explained:.1%}")
if 'deep_sleep_ratio' in available_vars and 'sleep_hours' in available_vars:
    print(f"  ✅ 驗證了PCA發現的睡眠品質權衡關係")
print(f"📊 可用於後續聚類分析和迴歸建模")